# 04 最終レポート

## 出力
1. 最も勝率が高い条件
2. 最もリターンが高い条件
3. 最も安定している条件
4. 現在その条件に当てはまる銘柄
5. 売買ルール
6. 改善提案

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
%matplotlib inline

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')

In [ ]:
# 全分析結果を読み込み
df = pd.read_pickle('../quant_research/data/_intermediate_df_features.pkl')
opt_results = pd.read_pickle('../quant_research/data/_results_optimization_results.pkl')

ml_results = None
try:
    ml_results = pd.read_pickle('../quant_research/data/_results_ml_results.pkl')
except FileNotFoundError:
    print("ML結果が見つかりません。Notebook 03を先に実行してください。")

regime_results = None
regime_summary = None
current_regime = 'unknown'
try:
    regime_results = pd.read_pickle('../quant_research/data/_results_regime_results.pkl')
    regime_summary = pd.read_pickle('../quant_research/data/_intermediate_regime_summary.pkl')
    current_regime = pd.read_pickle('../quant_research/data/_results_current_regime.pkl')
except FileNotFoundError:
    print("レジーム結果が見つかりません。")

print(f"最適化結果: {len(opt_results.get('all', [])):,} conditions")
print(f"現在のレジーム: {current_regime}")

## 最終レポート生成

In [ ]:
from quant_research.reporter import generate_final_report
from quant_research.screener import condition_to_str

report = generate_final_report(
    optimization_results=opt_results,
    ml_results=ml_results or {},
    regime_results=regime_results or {},
    regime_summary=regime_summary,
    df=df,
    current_regime=current_regime,
)

## 1. 最も勝率が高い条件

In [ ]:
bw = report['best_winrate']
print(f"条件: {bw['condition_str']}")
print(f"")
m = bw['metrics']
print(f"勝率:             {m['win_rate']:.1%}")
print(f"平均リターン:     {m['avg_return']:.2%}")
print(f"シャープレシオ:   {m['sharpe_ratio']:.2f}")
print(f"最大ドローダウン: {m['max_drawdown']:.1%}")
print(f"プロフィットF:    {m['profit_factor']:.2f}")
print(f"トレード数:       {m['n_trades']:,}")

## 2. 最もリターンが高い条件

In [ ]:
br = report['best_return']
print(f"条件: {br['condition_str']}")
print(f"")
m = br['metrics']
print(f"勝率:             {m['win_rate']:.1%}")
print(f"平均リターン:     {m['avg_return']:.2%}")
print(f"シャープレシオ:   {m['sharpe_ratio']:.2f}")
print(f"最大ドローダウン: {m['max_drawdown']:.1%}")
print(f"トレード数:       {m['n_trades']:,}")

## 3. 最も安定している条件

In [ ]:
bs = report['best_stable']
print(f"条件: {bs['condition_str']}")
print(f"")
m = bs['metrics']
print(f"勝率:             {m['win_rate']:.1%}")
print(f"平均リターン:     {m['avg_return']:.2%}")
print(f"シャープレシオ:   {m['sharpe_ratio']:.2f}")
print(f"最大ドローダウン: {m['max_drawdown']:.1%}")
print(f"プロフィットF:    {m['profit_factor']:.2f}")

## 4. 現在シグナルに当てはまる銘柄

In [ ]:
signals = report.get('current_signals', [])
print(f"現在のシグナル銘柄数: {len(signals)}")

if signals:
    signals_df = pd.DataFrame(signals)
    display(signals_df[[
        'code', 'name', 'close', 'vol_ratio', 'vol_zscore',
        'turnover', 'sector', 'market', 'signal_type'
    ]].head(30))

## 5. 売買ルール

In [ ]:
for key, rule in report['trading_rules'].items():
    print(f"\n{'='*50}")
    print(f"【{rule['name']}】")
    print(f"{'='*50}")
    print(f"エントリー: {rule['entry']}")
    print(f"条件:       {rule['conditions']}")
    print(f"決済:       {rule['exit']}")
    print(f"期待勝率:   {rule['expected_winrate']}")
    print(f"期待リターン: {rule['expected_return']}")
    print(f"リスク管理: {rule['risk_management']}")

## 6. 改善提案

In [ ]:
for i, s in enumerate(report['suggestions'], 1):
    print(f"  {i}. {s}")

## MLモデルサマリー

In [ ]:
for name, summary in report.get('ml_summary', {}).items():
    print(f"\n{name}:")
    print(f"  AUC:       {summary['avg_auc']:.4f}")
    print(f"  Precision: {summary['avg_precision']:.4f}")
    print(f"  Recall:    {summary['avg_recall']:.4f}")
    print(f"  F1:        {summary['avg_f1']:.4f}")
    print(f"  Top 5特徴量:")
    for feat, imp in list(summary.get('top_features', {}).items())[:5]:
        print(f"    {feat}: {imp:.4f}")

## 可視化まとめ

In [ ]:
from pathlib import Path
from IPython.display import Image, display as ipy_display

report_dir = Path('../quant_research/data/reports')
images = sorted(report_dir.glob('*.png'))

print(f"生成された画像: {len(images)} 枚")
for img in images:
    print(f"\n--- {img.name} ---")
    ipy_display(Image(filename=str(img), width=800))